# 1015 30일차

## 1. BM25

질문에 든 단어가 문서에 들어 있는지로 점수를 매기는 키워드 검색 (`rag23_01_bm25`)

임베딩 모델 없이 단어 일치만 보므로, 고유명사·제품명처럼 정확히 같은 단어를 찾을 때 씀

1. `BM25Retriever.from_documents(docs)` : `Document` 리스트로 검색기 생성. 임베딩·벡터 DB 없이 만듦
2. `k` : 돌려줄 문서 수. 만든 뒤 속성으로 지정 (`bm25_retriever.k = 3`)
3. `invoke(query)` : 점수 높은 순으로 `Document` 리스트를 돌려줌

- `rank_bm25` 패키지가 있어야 함

#### cf) 띄어쓰기 토큰화

- 기본 토큰화가 `text.split()`이라 띄어쓰기 단위로 단어를 나눔
- 조사가 붙으면 다른 단어가 됨. `삼성전자의`와 `삼성전자는`은 일치하지 않음
- 일치하는 단어가 없는 문서는 0점. 0점끼리는 순서 기준이 없어 질문과 상관없는 문서도 `k`개 안에 들어옴

## 2. 벡터 검색

질문과 문서를 임베딩해 벡터 유사도로 찾는 검색 (`rag23_02_vector`)

단어가 정확히 같지 않아도 뜻이 가까우면 찾아서, 조사나 표현이 달라도 관련 문서가 위로 옴

1. `Chroma.from_documents(documents, embedding)` : 문서를 임베딩해 저장. 인자 이름은 `embedding`
2. `as_retriever(search_kwargs={"k": 3})` : 검색기로 바꿈
3. `invoke(query)` : 가까운 순으로 `Document` 리스트를 돌려줌

## 3. 하이브리드 검색

키워드 검색과 벡터 검색의 결과를 합쳐 순위를 다시 매기는 검색 (`rag23_03_hybrid`)

BM25는 단어가 같아야만 찾고, 벡터 검색은 뜻이 가까운지만 비교해 정확한 단어 일치를 놓칠 수 있어 둘을 같이 씀

1. `EnsembleRetriever(retrievers=[...])` : 검색기 여러 개를 하나로 묶음
2. `weights` : 검색기별 중요도. `retrievers`와 같은 순서로 적음

- 검색기마다 문서 점수 `weight / (순위 + 60)`을 구해서 더함 (Reciprocal Rank Fusion)
- BM25 점수와 벡터 유사도는 범위가 달라서 점수 대신 순위를 씀
- 내용이 같은 문서는 하나로 합치고 점수를 더함. 두 검색기 모두 상위인 문서가 맨 위로 옴

#### cf) 결과 개수

- 하이브리드 검색에는 `k`를 따로 주지 않음
- 결과 개수는 검색기별 `k`의 합에서 겹친 문서 수를 뺀 만큼